# BodyBeats: custom face-expression model

Trains a small expression model to replace face-api's `faceExpressionNet` in `laptop/web/live.html`.

* **Data:** FER2013 faces (48x48 greyscale, from Kaggle) with Microsoft's **FER+** labels (10 people voted on every face; MIT licence).
* **Output:** the same 7 labels as face-api (`neutral, happy, sad, angry, fearful, disgusted, surprised`), so `fusion.js` and the UNO Q need no changes.
* **Benchmark:** the custom model and face-api on the **same** FER+ test faces: 7-class accuracy, accuracy on the 3 key groups the band actually uses, and how often the two agree.
* **Export:** `expression_model.json` + `expression_model.bin` (8-bit weights) + `expression_model_check.json`, read by `live/expression.js` using only basic tf.js ops.

**Before you start:** *Runtime > Change runtime type > T4 GPU*, then *Runtime > Run all*. The download cell asks you to sign in to Kaggle once.

In [ ]:
!pip -q install kagglehub
import os, json, glob, time, urllib.request
import numpy as np, pandas as pd, tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none (Runtime > Change runtime type > T4 GPU)")
tf.keras.utils.set_random_seed(7)

## 1. Data: FER2013 pictures + FER+ votes

In [ ]:
import kagglehub
def find_fer_csv():
    try:
        p = kagglehub.dataset_download("deadskull7/fer2013")
    except Exception as e:
        print("Kaggle needs you to sign in:", e)
        kagglehub.login()
        p = kagglehub.dataset_download("deadskull7/fer2013")
    hits = glob.glob(os.path.join(p, "**", "fer2013.csv"), recursive=True)
    if hits: return hits[0]
    from google.colab import files   # fallback: upload fer2013.csv yourself
    print("fer2013.csv not found in the download: upload it"); files.upload(); return "fer2013.csv"

fer = pd.read_csv(find_fer_csv())
urllib.request.urlretrieve("https://raw.githubusercontent.com/microsoft/FERPlus/master/fer2013new.csv", "fer2013new.csv")
plus = pd.read_csv("fer2013new.csv")
assert len(fer) == len(plus) == 35887, (len(fer), len(plus))
print(plus.head(3))

In [ ]:
# face-api's 7 labels, in the order the custom model outputs them
LABELS = ["neutral", "happy", "sad", "angry", "fearful", "disgusted", "surprised"]
# 3 key groups used by fusion.js: day = C major, dusk = D dorian, night = A minor
GROUP_OF = np.array([1, 0, 2, 2, 2, 2, 0])   # neutral->dusk, happy/surprised->day, sad/angry/fearful/disgusted->night
GROUPS = ["day (C major)", "dusk (D dorian)", "night (A minor)"]

votes = np.stack([plus.neutral, plus.happiness, plus.sadness, plus.anger, plus.fear,
                  plus.disgust + plus.contempt,      # FER+ 'contempt' is rare: counted as disgusted
                  plus.surprise], 1).astype(np.float32)
junk = np.maximum(plus.unknown, plus.NF).to_numpy()
keep = plus["Image name"].notna().to_numpy() & (votes.sum(1) > 0) & (votes.max(1) > junk)
X = np.stack([np.array(s.split(), dtype=np.uint8) for s in fer.pixels]).reshape(-1, 48, 48)
Y = votes / np.maximum(votes.sum(1, keepdims=True), 1)   # soft labels: the share of the votes
use = plus.Usage.to_numpy()
split = {k: keep & (use == u) for k, u in [("train", "Training"), ("val", "PublicTest"), ("test", "PrivateTest")]}
Xtr, Ytr = X[split["train"]], Y[split["train"]]
Xva, Yva = X[split["val"]], Y[split["val"]]
Xte, Yte = X[split["test"]], Y[split["test"]]
print({k: int(v.sum()) for k, v in split.items()}, "dropped (no face / unclear):", int((~keep).sum()))
print("train label mix:", dict(zip(LABELS, np.round(np.bincount(Ytr.argmax(1), minlength=7) / len(Ytr), 3))))

## 2. Model: a small depthwise-separable CNN (about 110k weights)

In [ ]:
L = tf.keras.layers
def conv_bn(x, f, name):
    x = L.Conv2D(f, 3, padding="same", use_bias=False, name=name)(x)
    return L.ReLU(name=name + "_relu")(L.BatchNormalization(name=name + "_bn")(x))
def sep(x, f, name):
    x = L.DepthwiseConv2D(3, padding="same", use_bias=False, name=name + "_dw")(x)
    x = L.ReLU(name=name + "_dw_relu")(L.BatchNormalization(name=name + "_dw_bn")(x))
    x = L.Conv2D(f, 1, use_bias=False, name=name + "_pw")(x)
    return L.ReLU(name=name + "_pw_relu")(L.BatchNormalization(name=name + "_pw_bn")(x))

def build():
    inp = L.Input((48, 48, 1), name="face")
    x = conv_bn(inp, 24, "c1"); x = conv_bn(x, 32, "c2"); x = L.MaxPool2D(name="p1")(x)    # 24x24
    x = sep(x, 64, "b1a");  x = sep(x, 64, "b1b");  x = L.MaxPool2D(name="p2")(x)          # 12x12
    x = sep(x, 128, "b2a"); x = sep(x, 128, "b2b"); x = L.MaxPool2D(name="p3")(x)          # 6x6
    x = sep(x, 192, "b3a"); x = sep(x, 192, "b3b")
    x = L.GlobalAveragePooling2D(name="gap")(x)
    x = L.Dropout(0.3, name="drop")(x)
    return tf.keras.Model(inp, L.Dense(7, name="head")(x), name="bodyband_expr")   # logits
model = build(); print("weights:", model.count_params())

In [ ]:
aug = tf.keras.Sequential([
    L.RandomFlip("horizontal"), L.RandomRotation(0.06), L.RandomZoom(0.15),
    L.RandomTranslation(0.08, 0.08), L.RandomContrast(0.35), L.RandomBrightness(0.25, value_range=(0, 1)),
])
def ds(x, y, train):
    d = tf.data.Dataset.from_tensor_slices((x[..., None], y))
    d = d.map(lambda a, b: (tf.cast(a, tf.float32) / 255.0, b))       # same scaling as live/expression.js
    if train:
        d = d.shuffle(20000).batch(128).map(lambda a, b: (tf.clip_by_value(aug(a, training=True), 0, 1), b),
                                           num_parallel_calls=tf.data.AUTOTUNE)
    else:
        d = d.batch(256)
    return d.prefetch(tf.data.AUTOTUNE)

EPOCHS = 60
steps = EPOCHS * int(np.ceil(len(Xtr) / 128))
opt = tf.keras.optimizers.AdamW(tf.keras.optimizers.schedules.CosineDecay(0.0, steps, warmup_target=2e-3, warmup_steps=steps // 30),
                                weight_decay=1e-4)
model.compile(opt, tf.keras.losses.CategoricalCrossentropy(from_logits=True), metrics=["categorical_accuracy"])
ckpt = tf.keras.callbacks.ModelCheckpoint("best.weights.h5", monitor="val_categorical_accuracy", save_best_only=True, save_weights_only=True)
t0 = time.time()
hist = model.fit(ds(Xtr, Ytr, True), validation_data=ds(Xva, Yva, False), epochs=EPOCHS, callbacks=[ckpt], verbose=2)
model.load_weights("best.weights.h5")
print("training took %.1f min, best val accuracy %.3f" % ((time.time() - t0) / 60, max(hist.history["val_categorical_accuracy"])))

## 3. How good is it? The custom model on the FER+ test faces

In [ ]:
def ours(x, flip=True):
    x = x[..., None].astype(np.float32) / 255.0
    p = tf.nn.softmax(model.predict(x, batch_size=512, verbose=0)).numpy()
    if flip: p = (p + tf.nn.softmax(model.predict(x[:, :, ::-1], batch_size=512, verbose=0)).numpy()) / 2
    return p

def report(name, p, y=None):
    y = Yte if y is None else y
    t7 = y.argmax(1); t3 = GROUP_OF[t7]
    g = np.stack([p[:, GROUP_OF == k].sum(1) for k in range(3)], 1)
    a7, a3 = (p.argmax(1) == t7).mean(), (g.argmax(1) == t3).mean()
    per = {LABELS[k]: round(float((p.argmax(1)[t7 == k] == k).mean()), 2) for k in range(7) if (t7 == k).any()}
    print(f"{name:28s} 7 expressions: {a7:.3f}   3 key groups: {a3:.3f}")
    print("   per expression:", per)
    return {"acc7": float(a7), "acc3": float(a3), "per_class": per}

P_ours_noflip, P_ours = ours(Xte, flip=False), ours(Xte)
results = {"ours (one pass)": report("ours (one pass)", P_ours_noflip), "ours (+ mirrored)": report("ours (+ mirrored)", P_ours)}
FLIP_TTA = results["ours (+ mirrored)"]["acc3"] > results["ours (one pass)"]["acc3"] + 0.003
P_best = P_ours if FLIP_TTA else P_ours_noflip
print("use the mirrored second pass in the browser:", FLIP_TTA)

## 4. face-api on the same test faces

Runs face-api's own `faceExpressionNet` (the one in `live/vendor/`) in Node on the same FER+ test faces. Both models get the same crop, so this compares only the expression step.

Two things to keep in mind. face-api was trained on bigger colour faces, so it is a little handicapped on 48x48 grey faces. And nobody knows whether FER2013 was in its training data. The live side-by-side test on real faces (section 7) is the second, fairer check.

In [ ]:
Xte.astype(np.uint8).tofile("test_u8.bin")
open("faceapi_bench.js", "w").write(r'''
const tf = require("@tensorflow/tfjs-node");
const faceapi = require("@vladmandic/face-api/dist/face-api.node.js");
const fs = require("fs");
const LABELS = ["neutral", "happy", "sad", "angry", "fearful", "disgusted", "surprised"];
(async () => {
  await faceapi.nets.faceExpressionNet.loadFromDisk("node_modules/@vladmandic/face-api/model");
  const buf = fs.readFileSync("test_u8.bin"), n = buf.length / 2304, out = [];
  const t0 = Date.now();
  for (let i = 0; i < n; i++) {
    const g = tf.tensor3d(Float32Array.from(buf.subarray(i * 2304, (i + 1) * 2304)), [48, 48, 1]);
    const rgb = tf.tile(g, [1, 1, 3]);
    let ex = await faceapi.nets.faceExpressionNet.predictExpressions(rgb);
    if (Array.isArray(ex)) ex = ex[0];
    out.push(LABELS.map((k) => ex[k]));
    tf.dispose([g, rgb]);
  }
  console.log("face-api: " + n + " faces in " + ((Date.now() - t0) / 1000).toFixed(1) + " s");
  fs.writeFileSync("faceapi_test.json", JSON.stringify(out));
})();
''')
!npm install --silent --no-audit --no-fund @vladmandic/face-api@1.7.13 @tensorflow/tfjs-node@4.16.0 > /dev/null 2>&1
!node faceapi_bench.js
try:
    P_fa = np.array(json.load(open("faceapi_test.json")), dtype=np.float32)
    results["face-api"] = report("face-api", P_fa)
    grp = lambda p: np.stack([p[:, GROUP_OF == k].sum(1) for k in range(3)], 1).argmax(1)
    results["agreement_3_groups"] = float((grp(P_best) == grp(P_fa)).mean())
    print(f"\nours and face-api pick the same key group on {results['agreement_3_groups']:.1%} of test faces")
except FileNotFoundError:
    print("face-api did not run in Node here (see the npm/node output above). The rest of the notebook still works.")

## 5. Export for the browser (8-bit weights, batch-norm folded in)

In [ ]:
def fold(conv_layer, bn_layer, depthwise):
    w = conv_layer.get_weights()[0]
    gamma, beta, mean, var = bn_layer.get_weights()
    s = gamma / np.sqrt(var + bn_layer.epsilon)
    w = w * (s[None, None, :, None] if depthwise else s[None, None, None, :])
    return w.astype(np.float32), (beta - mean * s).astype(np.float32)

blob = bytearray()
def put(a, quant):
    a = np.ascontiguousarray(a, dtype=np.float32)
    while len(blob) % 4: blob.append(0)
    d = {"shape": list(a.shape), "offset": len(blob)}
    if quant:
        lo, hi = float(a.min()), float(a.max()); sc = (hi - lo) / 255 or 1.0
        blob.extend(np.round((a - lo) / sc).clip(0, 255).astype(np.uint8).tobytes()); d.update(dtype="uint8", min=lo, scale=sc)
    else:
        blob.extend(a.tobytes()); d.update(dtype="float32")
    return d

ops = []
for lay in model.layers:
    n = lay.name
    if isinstance(lay, L.DepthwiseConv2D):
        w, b = fold(lay, model.get_layer(n + "_bn"), True)
        ops.append({"op": "dwconv", "name": n, "w": put(w, True), "b": put(b, False)})
    elif isinstance(lay, L.Conv2D):
        w, b = fold(lay, model.get_layer(n + "_bn"), False)
        ops.append({"op": "conv", "name": n, "w": put(w, True), "b": put(b, False)})
    elif isinstance(lay, L.MaxPool2D): ops.append({"op": "maxpool", "name": n})
    elif isinstance(lay, L.GlobalAveragePooling2D): ops.append({"op": "gap", "name": n})
    elif isinstance(lay, L.Dense):
        w, b = lay.get_weights(); ops.append({"op": "dense", "name": n, "w": put(w, True), "b": put(b, False)})
manifest = {"format": "bodyband-expr-1", "input": [48, 48, 1], "scale": "pixel/255, grey = 0.299R+0.587G+0.114B",
            "labels": LABELS, "flip_tta": bool(FLIP_TTA), "ops": ops,
            "trained_on": "FER2013 images + FER+ labels (Microsoft, MIT)", "test_results": results}
os.makedirs("export", exist_ok=True)
open("export/expression_model.bin", "wb").write(bytes(blob))
json.dump(manifest, open("export/expression_model.json", "w"), indent=1)
print("model file: %.0f KB (face-api's expression model: 322 KB)" % (len(blob) / 1024))

In [ ]:
# Re-run the EXPORTED 8-bit model with the same basic ops the browser uses (conv + bias + ReLU, etc.)
def load_exported():
    raw = open("export/expression_model.bin", "rb").read(); m = json.load(open("export/expression_model.json"))
    def t(d):
        n = int(np.prod(d["shape"]))
        if d["dtype"] == "uint8": v = d["min"] + np.frombuffer(raw, np.uint8, n, d["offset"]).astype(np.float32) * d["scale"]
        else: v = np.frombuffer(raw, np.float32, n, d["offset"])
        return tf.constant(v.reshape(d["shape"]), tf.float32)
    return [{**o, "w": t(o["w"]) if "w" in o else None, "b": t(o["b"]) if "b" in o else None} for o in m["ops"]]
def run_exported(ops, x):
    x = tf.constant(x[..., None].astype(np.float32) / 255.0)
    for o in ops:
        if o["op"] == "conv": x = tf.nn.relu(tf.nn.conv2d(x, o["w"], 1, "SAME") + o["b"])
        elif o["op"] == "dwconv": x = tf.nn.relu(tf.nn.depthwise_conv2d(x, o["w"], [1, 1, 1, 1], "SAME") + o["b"])
        elif o["op"] == "maxpool": x = tf.nn.max_pool2d(x, 2, 2, "VALID")
        elif o["op"] == "gap": x = tf.reduce_mean(x, [1, 2])
        elif o["op"] == "dense": x = tf.matmul(x, o["w"]) + o["b"]
    return tf.nn.softmax(x).numpy()
ops_x = load_exported()
P_x = (run_exported(ops_x, Xte) + run_exported(ops_x, Xte[:, :, ::-1])) / 2 if FLIP_TTA else run_exported(ops_x, Xte)
results["ours (exported, 8-bit)"] = report("ours (exported, 8-bit)", P_x)
print("largest change from 8-bit weights:", float(np.abs(P_x - P_best).max()))

# a few faces + expected answers (one pass, no mirror): live/expression.js checks itself against these
pick = np.random.RandomState(1).choice(len(Xte), 8, replace=False)
json.dump({"faces": [Xte[i].flatten().tolist() for i in pick], "expected": run_exported(ops_x, Xte[pick]).round(5).tolist()},
          open("export/expression_model_check.json", "w"))
manifest["test_results"] = results; json.dump(manifest, open("export/expression_model.json", "w"), indent=1)

## 6. Summary and download

In [ ]:
print(f"{'':28s}{'7 expressions':>15s}{'3 key groups':>15s}")
for k, v in results.items():
    if isinstance(v, dict): print(f"{k:28s}{v['acc7']:>15.3f}{v['acc3']:>15.3f}")
if "agreement_3_groups" in results: print("agreement with face-api on the key group: %.1f%%" % (100 * results["agreement_3_groups"]))
!cd export && zip -q -r ../bodyband_expression_model.zip . && ls -l
from google.colab import files; files.download("bodyband_expression_model.zip")

## 7. Next: the live test

Unzip into `laptop/web/live/model/`, open `live.html`, set **Expression model** to **Compare** and make faces at the Nicla camera or the webcam. The page shows both models' answers side by side and how often they pick the same key. Only switch to **Ours** once it holds up there too.